In [ ]:
import h5py
import numpy as np

exp_path = './exp.h5'
exp_reco_path = './exp_reco.h5'

In [ ]:
with h5py.File(exp_path) as he:
    with h5py.File(exp_reco_path) as hr:
        e_group = he['exp']
        r_group = hr['exp']
        
        parts = list(r_group['clusters_centers'].keys())
        part_name = parts[-1]
        print(part_name)
        
        print(he[f'exp/clusters_centers/{part_name}/data'][:])
        print(hr[f'exp/clusters_centers/{part_name}/data'][:])
        
        print(e_group['coords_are_cluster_centered/data'][()])
        print(r_group['coords_are_cluster_centered/data'][()])
        
        print(e_group[f'raw/channels/{part_name}/data'][:])
        print(r_group[f'raw/channels/{part_name}/data'][:])
        
        # First reco event
        r_ev_starts = r_group[f"raw/ev_starts/{part_name}/data"][:]
        r_s, r_e = r_ev_starts[0], r_ev_starts[1]
        first_reco_event_ch = r_group[f'raw/channels/{part_name}/data'][r_s:r_e]
        r_t = r_group[f'raw/data/{part_name}/data'][r_s:r_e, 1]
        r_z = r_group[f'raw/data/{part_name}/data'][r_s:r_e, 4]
        
        
        # Find match
        e_ev_starts = e_group[f"raw/ev_starts/{part_name}/data"][:]
        for e_s, e_e in zip(e_ev_starts[:-1], e_ev_starts[1:]):
            e_channels = e_group[f'raw/channels/{part_name}/data'][e_s:e_e]
            e_t = r_group[f'raw/data/{part_name}/data'][e_s:e_e, 1]
            e_z = r_group[f'raw/data/{part_name}/data'][e_s:e_e, 4]
            if np.array_equal(e_channels, first_reco_event_ch):
                print(f"Found event! Hits num {e_e-e_s}")
                
                
                print(f"Z mean: {r_z.mean()} VS {e_z.mean()}")
                break

In [ ]:
first_reco_event_ch

In [ ]:
first_reco_event_ch, e_channels

In [ ]:
e_z[0], r_z[0]

In [ ]:
import uproot as ur

r_rf_path = "/home/albert/Baikal2025/data_manager/exp_reco_root/root_files/2020_cl7_run406_scl_nu_DATA2020.root"
with ur.open(r_rf_path) as rf:
    e_channels = rf['Events/BEvent./BEvent.fPulses/BEvent.fPulses.fChannelID'].array(library='np')[1:]
    active_clusters = [np.unique(ch // 288) for ch in e_channels]
    num_un_clusters = np.array([len(cl) for cl in active_clusters])
    
print(e_channels)

In [ ]:
import awkward as ak
e_rf_path = "/net/62/home/albert/Baikal/Data/exp_root_files/s2020_c07_r0406.root"
path_geometry = "Events/BGeomTel./BGeomTel.BGeomTel/BGeomTel.BGeomTel.fOMs/BGeomTel.BGeomTel.fOMs.fPosition"
st = 0

with ur.open(e_rf_path) as rf:
    e_channels = rf['Events/BEvent./BEvent.fPulses/BEvent.fPulses.fChannelID'].array(library='np')[st:]
    coordinates = np.array(ak.unzip(rf[path_geometry].array()))[:,st:]
    active_clusters = [np.unique(ch // 288) for ch in e_channels]
    num_un_clusters = np.array([len(cl) for cl in active_clusters])
    
e_channels

In [ ]:
coordinates.shape

In [ ]:
e_channels, e_z